In [1]:
import pandas as pd
import numpy as np
import ipywidgets as widgets
from bin_model import BinomialModel
from options import EuropeanCall, EuropeanPut


S0 = 50.0      
K = 48.0       
T = 2.0        
r = 0.02       
sigma = 0.3    
dt = 1/12      
N = int(T / dt)                
u = np.exp(sigma * np.sqrt(dt)) 
d = 1 / u                       

def display_hedging_results(t_selected, option_type):
    model = BinomialModel(S0, N, dt, u, d, r)
    opt = EuropeanCall(K) if option_type == 'Call' else EuropeanPut(K)
    
    # obliczanie wartości dla każdęgo węzła
    opt_tree, delta_tree, alpha_tree = opt.price_with_hedging(model)
    stock_tree = model.generate_stock_tree()
    
    # budowa danych do tabeli dla wybranego kroku czasowego t
    rows = []
    for i in range(t_selected + 1):
        # wyciągamy dane z drzewa dla danej chwili w czasie
        s_val = stock_tree[i, t_selected]
        v_val = opt_tree[i, t_selected]
        
        # delata i alfa pod koniec nie da sięwyliczyć wrzycamy elsa na zakończenie hedgingu
        if t_selected < N:
            d_val = delta_tree[i, t_selected]
            a_val = alpha_tree[i, t_selected]
            # Weryfikacja czy protfel jest warty ile węzeł w chwili t: delta * S + alpha
            portfolio_val = d_val * s_val + a_val
        else:
            d_val, a_val, portfolio_val = None, None, v_val

        rows.append({
            "Węzeł (i,t)": f"({i}, {t_selected})",
            "Cena Akcji S_t": s_val,
            "Wartość Opcji V_t": v_val,
            "Delta ": d_val,
            "Alpha ": a_val,
            "Portfel Replikujący": portfolio_val
        })
    
    # wyświetladnie w pandas
    df = pd.DataFrame(rows)
    
    
    styled_df = df.style.format({
        "Cena Akcji S_t": "{:.2f}",
        "Wartość Opcji V_t": "{:.4f}",
        "Delta ": "{:.4f}",
        "Alpha ": "{:.4f}",
        "Portfel Replikujący": "{:.4f}"
    }, na_rep="---").background_gradient(subset=["Delta "], cmap="RdYlGn", vmin=-1, vmax=1)
    
    display(styled_df)

# suwak 
widgets.interact(
    display_hedging_results,
    t_selected = widgets.IntSlider(
        value=0, 
        min=0, 
        max=N, 
        step=1, 
        description='Krok t:',
        style={'description_width': 'initial'}
    ),
    option_type = widgets.Dropdown(
        options=['Call', 'Put'],
        value='Call',
        description='Typ opcji:'
    )
);

interactive(children=(IntSlider(value=0, description='Krok t:', max=24, style=SliderStyle(description_width='i…